# Positional Embedding Experiments

Core ML Task 3 notebook. This is based on the baseline Transformer, not on the attention-variant notebook. It trains each positional encoding at context length 512 and evaluates extrapolation at 512, 1024, and 2048. Optional positional interpolation/scaling is intentionally excluded.

# Setup

In [ ]:
!pip install -q datasets transformers tqdm pandas

# Imports

In [ ]:
import math
import os
import random
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Drive Logging

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/SAiDL_Assignment/core_ml_positional_embeddings"
except Exception:
    LOG_DIR = "positional_embedding_logs"

os.makedirs(LOG_DIR, exist_ok=True)
print("Logging to:", LOG_DIR)

# Config

In [ ]:
class Config:
    vocab_size = 50257
    max_block_size = 2048
    train_block_size = 512
    n_layer = 4
    n_head = 4
    n_embd = 256
    dropout = 0.1
    learning_rate = 3e-4

    # Extrapolation test required by the assignment.
    eval_context_lengths = [512, 1024, 2048]

    # Keep approximately similar token count per batch where possible.
    batch_sizes = {
        512: 16,
        1024: 8,
        2048: 4,
    }

    epochs = 15
    seed = 42

    # Relative position bias clips all longer distances into one final bucket.
    max_relative_distance = train_block_size - 1


def make_config(position_mode):
    return SimpleNamespace(
        vocab_size=Config.vocab_size,
        block_size=Config.max_block_size,
        train_block_size=Config.train_block_size,
        n_layer=Config.n_layer,
        n_head=Config.n_head,
        n_embd=Config.n_embd,
        dropout=Config.dropout,
        learning_rate=Config.learning_rate,
        position_mode=position_mode,
        max_relative_distance=Config.max_relative_distance,
    )

# Load Dataset + Tokenizer

In [ ]:
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Tokenize Once

In [ ]:
def tokenize(example):
    return tokenizer(example["text"])


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)

# Build Context-Length Datasets + DataLoaders

In [ ]:
lm_dataset_cache = {}
dataloader_cache = {}


def build_lm_dataset(block_size):
    if block_size in lm_dataset_cache:
        return lm_dataset_cache[block_size]

    def group_texts(examples):
        concatenated = sum(examples["input_ids"], [])
        total_length = (len(concatenated) // block_size) * block_size

        input_ids = [
            concatenated[i:i + block_size]
            for i in range(0, total_length, block_size)
        ]

        return {"input_ids": input_ids, "labels": input_ids.copy()}

    lm_datasets = tokenized.map(
        group_texts,
        batched=True,
        remove_columns=tokenized["train"].column_names,
    )
    lm_dataset_cache[block_size] = lm_datasets
    return lm_datasets


def collate(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch], dtype=torch.long)
    labels = torch.tensor([x["labels"] for x in batch], dtype=torch.long)
    return input_ids, labels


def build_dataloaders(block_size, batch_size, shuffle_train=True):
    cache_key = (block_size, batch_size, shuffle_train)
    if cache_key in dataloader_cache:
        return dataloader_cache[cache_key]

    lm_datasets = build_lm_dataset(block_size)

    train_loader = torch.utils.data.DataLoader(
        lm_datasets["train"],
        batch_size=batch_size,
        shuffle=shuffle_train,
        collate_fn=collate,
    )

    val_loader = torch.utils.data.DataLoader(
        lm_datasets["validation"],
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate,
    )

    dataloader_cache[cache_key] = (train_loader, val_loader)
    return train_loader, val_loader

# Positional Encoding Helpers

In [ ]:
def rotate_half_interleaved(x):
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)


def apply_rope(q, k):
    # q/k shape: (B, H, T, D)
    T = q.size(-2)
    D = q.size(-1)
    assert D % 2 == 0

    inv_freq = 1.0 / (10000 ** (torch.arange(0, D, 2, device=q.device).float() / D))
    positions = torch.arange(T, device=q.device).float()
    freqs = torch.einsum("t,d->td", positions, inv_freq)

    cos = freqs.cos().repeat_interleave(2, dim=-1).view(1, 1, T, D)
    sin = freqs.sin().repeat_interleave(2, dim=-1).view(1, 1, T, D)

    q_rot = (q * cos) + (rotate_half_interleaved(q) * sin)
    k_rot = (k * cos) + (rotate_half_interleaved(k) * sin)
    return q_rot, k_rot


def get_alibi_slopes(n_heads):
    def get_slopes_power_of_2(n):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(n_heads).is_integer():
        return torch.tensor(get_slopes_power_of_2(n_heads), dtype=torch.float32)

    closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
    slopes = get_slopes_power_of_2(closest_power_of_2)
    extra = get_alibi_slopes(2 * closest_power_of_2)[0::2]
    slopes.extend(extra[: n_heads - closest_power_of_2].tolist())
    return torch.tensor(slopes, dtype=torch.float32)

# Attention With Positional Variants

In [ ]:
class PositionalCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.config = config
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.position_mode = config.position_mode

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size)).bool()
        self.register_buffer("causal_mask", causal_mask.unsqueeze(0).unsqueeze(0))

        if self.position_mode == "alibi":
            slopes = get_alibi_slopes(config.n_head).view(1, config.n_head, 1, 1)
            positions = torch.arange(config.block_size)
            distance = (positions[:, None] - positions[None, :]).float()
            alibi_bias = -slopes * distance.view(1, 1, config.block_size, config.block_size)
            self.register_buffer("alibi_bias", alibi_bias)

        if self.position_mode == "relative_bias":
            self.relative_bias = nn.Embedding(config.max_relative_distance + 1, config.n_head)

    def get_relative_bias(self, T, device):
        positions = torch.arange(T, device=device)
        distance = positions[:, None] - positions[None, :]
        distance = distance.clamp(min=0, max=self.config.max_relative_distance)
        bias = self.relative_bias(distance)
        return bias.permute(2, 0, 1).unsqueeze(0)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if self.position_mode == "rope":
            q, k = apply_rope(q, k)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if self.position_mode == "alibi":
            att = att + self.alibi_bias[:, :, :T, :T]
        elif self.position_mode == "relative_bias":
            att = att + self.get_relative_bias(T, x.device)

        att = att.masked_fill(~self.causal_mask[:, :, :T, :T], float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

# Feedforward + Transformer Block

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = PositionalCausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# Full Model

In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.position_mode = config.position_mode

        self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)

        if self.position_mode == "learned_absolute":
            self.pos_emb = nn.Parameter(torch.zeros(1, config.block_size, config.n_embd))
        else:
            self.pos_emb = None

        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size

        x = self.token_emb(idx)

        if self.position_mode == "learned_absolute":
            x = x + self.pos_emb[:, :T, :]

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, logits.size(-1)),
                targets[:, 1:].contiguous().view(-1),
            )

        return logits, loss

# Training + Evaluation Helpers

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def reset_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def get_peak_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0.0


def evaluate(model, val_loader):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, loss = model(x, y)
            losses.append(loss.item())

    model.train()
    return sum(losses) / len(losses)


def measure_inference_throughput(model, val_loader, max_batches=20):
    model.eval()
    total_tokens = 0

    sync_cuda()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(val_loader):
            if batch_idx >= max_batches:
                break
            x = x.to(device)
            model(x)
            total_tokens += x.numel()

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    model.train()

    return total_tokens / max(elapsed, 1e-9)


def save_checkpoint(path, model, optimizer, epoch, metrics):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics": metrics,
        },
        path,
    )

# Experiment Runner

In [ ]:
def train_positional_experiment(
    position_name,
    position_mode,
    epochs=Config.epochs,
    seed=Config.seed,
):
    set_seed(seed)
    config = make_config(position_mode)

    train_batch_size = Config.batch_sizes[Config.train_block_size]
    train_loader, _ = build_dataloaders(Config.train_block_size, train_batch_size)

    eval_loaders = {}
    for context_length in Config.eval_context_lengths:
        batch_size = Config.batch_sizes[context_length]
        _, val_loader = build_dataloaders(context_length, batch_size, shuffle_train=False)
        eval_loaders[context_length] = (batch_size, val_loader)

    model = TransformerModel(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

    run_id = f"{position_name}_trainctx{Config.train_block_size}_seed{seed}"
    metrics_path = os.path.join(LOG_DIR, f"{run_id}_metrics.csv")
    checkpoint_path = os.path.join(LOG_DIR, f"{run_id}_latest.pt")

    start_epoch = 0

    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        start_epoch = ckpt["epoch"] + 1

        print(f"Resuming from epoch {start_epoch}")

    if os.path.exists(metrics_path):
        all_metrics = pd.read_csv(metrics_path).to_dict("records")
        print(f"Loaded {len(all_metrics)} metric rows")
    else:
        all_metrics = []

    for epoch in range(start_epoch, epochs):
        model.train()
        reset_peak_memory()
        pbar = tqdm(train_loader, desc=f"{position_name} epoch {epoch}")

        total_train_loss = 0.0
        total_tokens = 0

        sync_cuda()
        start_time = time.perf_counter()

        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            _, loss = model(x, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_tokens = y[:, 1:].numel()
            total_tokens += batch_tokens
            total_train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        sync_cuda()
        epoch_time = time.perf_counter() - start_time

        train_loss = total_train_loss / len(train_loader)
        train_throughput = total_tokens / epoch_time
        train_peak_gpu_memory_mb = get_peak_memory_mb()

        epoch_rows = []
        for eval_context_length, (eval_batch_size, val_loader) in eval_loaders.items():
            reset_peak_memory()
            val_loss = evaluate(model, val_loader)
            val_perplexity = math.exp(min(val_loss, 20.0))
            inference_tokens_per_sec = measure_inference_throughput(model, val_loader)
            eval_peak_gpu_memory_mb = get_peak_memory_mb()

            row = {
                "position_name": position_name,
                "position_mode": position_mode,
                "seed": seed,
                "epoch": epoch,
                "train_context_length": Config.train_block_size,
                "eval_context_length": eval_context_length,
                "train_batch_size": train_batch_size,
                "eval_batch_size": eval_batch_size,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_perplexity": val_perplexity,
                "epoch_time_sec": epoch_time,
                "train_throughput_tokens_per_sec": train_throughput,
                "inference_throughput_tokens_per_sec": inference_tokens_per_sec,
                "train_peak_gpu_memory_mb": train_peak_gpu_memory_mb,
                "eval_peak_gpu_memory_mb": eval_peak_gpu_memory_mb,
            }
            epoch_rows.append(row)
            all_metrics.append(row)

        pd.DataFrame(all_metrics).to_csv(metrics_path, index=False)
        save_checkpoint(checkpoint_path, model, optimizer, epoch, epoch_rows)

        print(f"Epoch {epoch} complete for {position_name}")
        display(pd.DataFrame(epoch_rows))
        print("saved metrics:", metrics_path)
        print("saved checkpoint:", checkpoint_path)

    return pd.DataFrame(all_metrics)

# Run: Learned Absolute Reference

In [ ]:
# learned_absolute_results = train_positional_experiment(
#     position_name="learned_absolute_reference",
#     position_mode="learned_absolute",
#     epochs=Config.epochs,
#     seed=Config.seed,
# )

# display(learned_absolute_results)

# Run: RoPE

In [ ]:
rope_results = train_positional_experiment(
    position_name="rope",
    position_mode="rope",
    epochs=Config.epochs,
    seed=Config.seed,
)

display(rope_results)

# Run: ALiBi

In [ ]:
# alibi_results = train_positional_experiment(
#     position_name="alibi",
#     position_mode="alibi",
#     epochs=Config.epochs,
#     seed=Config.seed,
# )

# display(alibi_results)

# Run: Relative Position Bias

In [ ]:
# relative_bias_results = train_positional_experiment(
#     position_name="relative_position_bias",
#     position_mode="relative_bias",
#     epochs=Config.epochs,
#     seed=Config.seed,
# )

# display(relative_bias_results)

# Aggregate Saved Results

In [ ]:
metric_files = [
    os.path.join(LOG_DIR, name)
    for name in os.listdir(LOG_DIR)
    if name.endswith("_metrics.csv")
]

if metric_files:
    all_metrics = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    display(all_metrics)

    final_epoch_metrics = all_metrics.sort_values("epoch").groupby(
        ["position_name", "eval_context_length", "seed"],
        as_index=False,
    ).tail(1)

    display(final_epoch_metrics.sort_values(["eval_context_length", "val_perplexity"]))

    aggregate_path = os.path.join(LOG_DIR, "positional_embeddings_all_metrics.csv")
    all_metrics.to_csv(aggregate_path, index=False)
    print("saved aggregate metrics:", aggregate_path)
else:
    print("No metric files found yet in", LOG_DIR)